In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_Goods_Receipts
# Source          : Goods_Receipts.csv
# Target          : procurement.silver.silver_Goods_Receipts
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned Goods_Receipts master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp, to_date, try_to_date

In [0]:
# ============================================================
# Read Bronze Invoice Table
# ============================================================

bronze_goods_receipts_df = read_delta(BRONZE_GOODS_RECEIPTS)

preview(bronze_goods_receipts_df,"Bronze goods_receipts")

In [0]:
#============================================
# Create Sliver DataFrame
#===========================================
silver_goods_receipts_df = bronze_goods_receipts_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================
silver_goods_receipts_df = (silver_goods_receipts_df

    # Trim string columns
    .withColumn("grn_id", trim(col("grn_id")))
    .withColumn("po_id", trim(col("po_id")))
    .withColumn("receipt_status", initcap(trim(col("receipt_status")))) 
    
    # Date columns
    .withColumn("grn_date",
                to_date(trim(col("grn_date")), "dd-MM-yyyy"))

    # Numeric columns (no trim)
    .withColumn("quantity_ordered_ref", col("quantity_ordered_ref").cast("int"))
    .withColumn("quantity_received", col("quantity_received").cast("int"))
    
    # Standardize received_by
    .withColumn(
        "received_by",
        when(
            upper(trim(col("received_by"))).isin("E0066", "E00 66"),
            "E00     66"
        )
        .when(
            trim(col("received_by")) == "",
            None
        )
        .otherwise(upper(trim(col("received_by"))))
    )
)



In [0]:
# ============================================================
# Identify Invalid Goods_Receipts Records
# NULL & Blank: grn_id
# NULL: grn_date,po_id ,received_by
# Zero: quantity_ordered_ref,quantity_received
# ============================================================

invalid_goods_receipts = (
    silver_goods_receipts_df.filter(

        # GRN ID
        col("grn_id").isNull()
        | (trim(col("grn_id")) == "")

        # GRN Date
        | col("grn_date").isNull()

        #PO
        | col("po_id").isNull()
        | (trim(col("po_id")) == "")

        # Quantity ORDERED REF
        | col("quantity_ordered_ref").isNull()
        | (col("quantity_ordered_ref") <= 0)

        # Quantity_Received
        | col("quantity_received").isNull()
        | (col("quantity_received") <= 0)

        # Receipt_Status
        | col("receipt_status").isNull()
        | (trim(col("receipt_status")) == "")

         # Received_By
        | col("received_by").isNull()
        | (trim(col("received_by")) == "")
    )
)

print(
    f"Invalid Goods_Receipts Records: "
    f"{invalid_goods_receipts.count()}"
)

display(invalid_goods_receipts)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_goods_receipts = (invalid_goods_receipts.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("goods_receipts"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_goods_receipts)

In [0]:
# ============================================================
# Write Invalid GOODS_RECEIPTS to Audit Table
# ============================================================

if invalid_goods_receipts.count() > 0:
    write_delta(invalid_goods_receipts,AUDIT_INVALID_GOODS_RECEIPTS,mode="overwrite")
    print("Invalid invoice records written.")
else:
    print("No invalid invoice records found.")

In [0]:

# ============================================================
# Remove Invalid Goods_Receipts Records
# ============================================================

silver_goods_receipts_df = silver_goods_receipts_df.filter(

    # GRN ID
    col("grn_id").isNotNull() &
    (trim(col("grn_id")) != "") &

    # GRN Date
    col("grn_date").isNotNull() &

    # PO ID
    col("po_id").isNotNull() &
    (trim(col("po_id")) != "") &

    # Quantity Ordered Ref
    col("quantity_ordered_ref").isNotNull() &
    (col("quantity_ordered_ref") > 0) &

    # Quantity Received
    col("quantity_received").isNotNull() &
    (col("quantity_received") > 0) &


    # Receipt_status
    col("receipt_status").isNotNull() &
    (trim(col("receipt_status")) != "") &

    # Received_By
    col("received_by").isNull()
    | (trim(col("received_by")) == "")

)

display(silver_goods_receipts_df)

In [0]:
# ============================================================
# Remove Duplicate GOODS_RECEIPTS
# ============================================================

window_spec = Window.partitionBy("grn_id").orderBy("grn_date")

silver_goods_receipts_df = (silver_goods_receipts_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_goods_receipts_df,"Silver Goods_Receipts")


In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_goods_receipts_df = (silver_goods_receipts_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_goods_receipts_df,table_name=SILVER_GOODS_RECEIPTS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

bronze_count = bronze_goods_receipts_df.count()
invalid_count = invalid_goods_receipts.count()
silver_count = silver_goods_receipts_df.count()

duplicate_removed = bronze_count - invalid_count - silver_count

print("=" * 60)
print("Silver goods_receipts Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records            : {bronze_count}")
print(f"Invalid Records Removed   : {invalid_count}")
print(f"Duplicate Records Removed : {duplicate_removed}")
print(f"Silver Records            : {silver_count}")